# WEEK8 知识精讲：pandas 缺失值、MultiIndex 与分组聚合

> **中文说明**：本 notebook 依据 `WEEK8/source/lec7_DataFrame_II.ipynb` 重写。主题包括：缺失值处理、MultiIndex（层次化索引）、`stack` / `unstack`、索引的重置与设置，以及 `groupby` 分组聚合。代码全部真实运行，输出已保存；练习答案默认折叠。

**使用建议**

1. 先理解“为什么需要这个功能”，再看最小例子。
2. 每个例子都先看 `shape` 和索引，再看数值。
3. 带 **注意** 的地方是作业和考试的高频坑。

## 目录

1. 数据准备与缺失值识别
2. 删除缺失值：`dropna`
3. 填充缺失值：`fillna` / `ffill` / `bfill`
4. 完整案例：订单数据五步处理
5. 为什么需要 MultiIndex
6. 创建 MultiIndex
7. MultiIndex 的选择与切片
8. 排序、`IndexSlice` 与复杂选择
9. `stack` / `unstack`：宽表与长表
10. 列的 MultiIndex
11. `reset_index` / `set_index`
12. `groupby` 基础
13. 多列分组与 `as_index`
14. `agg`：一次做多种聚合
15. 遍历 groupby 对象
16. 自定义分组键：mapping / function / MultiIndex
17. 练习：health_data 的层级交换与选择
18. 练习：sales 的 MultiIndex 构造
19. 练习：学生数据分组与 BMI
20. 速查表与常见坑

## 1. 数据准备与缺失值识别

pandas 用 `NaN`（数值）、`None`（对象）、`NaT`（时间）表示缺失值。统一用 `isna()` 检测最方便。

> 运行环境：pandas 3.x。pandas 3 默认启用 Copy-on-Write，`inplace=True` 仍然可用，但更推荐“赋值给新变量”的写法。

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("pandas version:", pd.__version__)

df = pd.DataFrame({
    'ord_no': [70001, np.nan, 70002, np.nan, np.nan, 70005,
               np.nan, np.nan, 70003, 70012, np.nan, 70013],
    'purch_amt': [150.5, 270.65, 65.26, 110.5, 948.5, 2400.6,
                  5760, 1983.43, 2480.4, 250.45, 75.29, 3045.6],
    'ord_date': ['2012-10-05', '2012-09-10', np.nan, '2012-08-17',
                 '2012-09-10', '2012-07-27', '2012-09-10', '2012-10-10',
                 '2012-10-10', '2012-06-27', '2012-08-17', '2012-04-25'],
    'customer_id': [3002, 3001, 3001, 3003, 3002, 3001,
                    3001, 3004, 3003, 3002, 3001, 3001],
    'salesman_id': [5002, 5003, 5001, np.nan, 5002, 5001,
                    5001, np.nan, 5003, 5002, 5003, np.nan]
})

print(df)

pandas version: 3.0.6
     ord_no  purch_amt    ord_date  customer_id  salesman_id
0   70001.0     150.50  2012-10-05         3002       5002.0
1       NaN     270.65  2012-09-10         3001       5003.0
2   70002.0      65.26         NaN         3001       5001.0
3       NaN     110.50  2012-08-17         3003          NaN
4       NaN     948.50  2012-09-10         3002       5002.0
5   70005.0    2400.60  2012-07-27         3001       5001.0
6       NaN    5760.00  2012-09-10         3001       5001.0
7       NaN    1983.43  2012-10-10         3004          NaN
8   70003.0    2480.40  2012-10-10         3003       5003.0
9   70012.0     250.45  2012-06-27         3002       5002.0
10      NaN      75.29  2012-08-17         3001       5003.0
11  70013.0    3045.60  2012-04-25         3001          NaN


In [2]:
print(df.isna())

    ord_no  purch_amt  ord_date  customer_id  salesman_id
0    False      False     False        False        False
1     True      False     False        False        False
2    False      False      True        False        False
3     True      False     False        False         True
4     True      False     False        False        False
5    False      False     False        False        False
6     True      False     False        False        False
7     True      False     False        False         True
8    False      False     False        False        False
9    False      False     False        False        False
10    True      False     False        False        False
11   False      False     False        False         True


In [3]:
print("每列缺失个数:")
print(df.isna().sum())
print("\n每列非缺失个数:")
print(df.notna().sum())

每列缺失个数:
ord_no         6
purch_amt      0
ord_date       1
customer_id    0
salesman_id    3
dtype: int64

每列非缺失个数:
ord_no          6
purch_amt      12
ord_date       11
customer_id    12
salesman_id     9
dtype: int64


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ord_no       6 non-null      float64
 1   purch_amt    12 non-null     float64
 2   ord_date     11 non-null     str    
 3   customer_id  12 non-null     int64  
 4   salesman_id  9 non-null      float64
dtypes: float64(3), int64(1), str(1)
memory usage: 612.0 bytes


**怎么看**：
- `isna()` 返回同形状的布尔表；
- `isna().sum()` 按列统计 `True` 的个数；
- `info()` 直接给出每列的非空数量和 dtype，是拿到新数据后的第一件事。

## 2. 删除缺失值：`dropna`

`dropna` 的常用参数：

| 参数 | 含义 |
|---|---|
| `axis=0` / `'index'` | 删行（默认） |
| `axis=1` / `'columns'` | 删列 |
| `how='any'` | 只要有一个缺失就删（默认） |
| `how='all'` | 全部缺失才删 |
| `thresh=n` | 至少保留 n 个**非缺失**值，否则删 |
| `subset=[...]` | 只看指定列 |

**关键**：`thresh` 数的是“非缺失值”的个数，不是缺失值的个数。

In [5]:
# 先去掉 ord_date 这一列
df1 = df.drop(columns='ord_date')       # 等价于 df.drop('ord_date', axis=1)
print(df1)

     ord_no  purch_amt  customer_id  salesman_id
0   70001.0     150.50         3002       5002.0
1       NaN     270.65         3001       5003.0
2   70002.0      65.26         3001       5001.0
3       NaN     110.50         3003          NaN
4       NaN     948.50         3002       5002.0
5   70005.0    2400.60         3001       5001.0
6       NaN    5760.00         3001       5001.0
7       NaN    1983.43         3004          NaN
8   70003.0    2480.40         3003       5003.0
9   70012.0     250.45         3002       5002.0
10      NaN      75.29         3001       5003.0
11  70013.0    3045.60         3001          NaN


In [6]:
print("每行缺失个数:")
print(df1.isna().sum(axis=1))

每行缺失个数:
0     0
1     1
2     0
3     2
4     1
5     0
6     1
7     2
8     0
9     0
10    1
11    1
dtype: int64


In [7]:
print("how='any' 删除后剩余行数:", len(df1.dropna(how='any')))
print("how='all' 删除后剩余行数:", len(df1.dropna(how='all')))

print("\nthresh=3（至少 3 个非缺失，即最多 1 个缺失）:")
print(df1.dropna(thresh=3))

how='any' 删除后剩余行数: 5
how='all' 删除后剩余行数: 12

thresh=3（至少 3 个非缺失，即最多 1 个缺失）:
     ord_no  purch_amt  customer_id  salesman_id
0   70001.0     150.50         3002       5002.0
1       NaN     270.65         3001       5003.0
2   70002.0      65.26         3001       5001.0
4       NaN     948.50         3002       5002.0
5   70005.0    2400.60         3001       5001.0
6       NaN    5760.00         3001       5001.0
8   70003.0    2480.40         3003       5003.0
9   70012.0     250.45         3002       5002.0
10      NaN      75.29         3001       5003.0
11  70013.0    3045.60         3001          NaN


**注意**：本数据有 4 列，`thresh=3` 表示“至少 3 个非缺失”，等价于“缺失值不超过 1 个”。如果写成 `thresh=1`，意思完全不同。

In [8]:
# 另一种等价写法：自己数每行缺失个数，再过滤
df2_alt = df1[df1.isna().sum(axis=1) <= 1]
print(df2_alt)

     ord_no  purch_amt  customer_id  salesman_id
0   70001.0     150.50         3002       5002.0
1       NaN     270.65         3001       5003.0
2   70002.0      65.26         3001       5001.0
4       NaN     948.50         3002       5002.0
5   70005.0    2400.60         3001       5001.0
6       NaN    5760.00         3001       5001.0
8   70003.0    2480.40         3003       5003.0
9   70012.0     250.45         3002       5002.0
10      NaN      75.29         3001       5003.0
11  70013.0    3045.60         3001          NaN


## 3. 填充缺失值：`fillna` / `ffill` / `bfill`

| 方法 | 作用 |
|---|---|
| `fillna(value)` | 用固定值填充 |
| `fillna(dict)` | 不同列填不同值 |
| `ffill()` | 用前一个有效值向下填（forward fill） |
| `bfill()` | 用后一个有效值向上填（backward fill） |
| `interpolate()` | 线性/多项式插值（连续数值常用） |

默认 `axis=0`，即沿着行方向（上下）填充。

In [9]:
df2 = df1.dropna(thresh=3)
print(df2)

     ord_no  purch_amt  customer_id  salesman_id
0   70001.0     150.50         3002       5002.0
1       NaN     270.65         3001       5003.0
2   70002.0      65.26         3001       5001.0
4       NaN     948.50         3002       5002.0
5   70005.0    2400.60         3001       5001.0
6       NaN    5760.00         3001       5001.0
8   70003.0    2480.40         3003       5003.0
9   70012.0     250.45         3002       5002.0
10      NaN      75.29         3001       5003.0
11  70013.0    3045.60         3001          NaN


In [10]:
print("ffill（前向填充）:")
print(df2.ffill())

ffill（前向填充）:
     ord_no  purch_amt  customer_id  salesman_id
0   70001.0     150.50         3002       5002.0
1   70001.0     270.65         3001       5003.0
2   70002.0      65.26         3001       5001.0
4   70002.0     948.50         3002       5002.0
5   70005.0    2400.60         3001       5001.0
6   70005.0    5760.00         3001       5001.0
8   70003.0    2480.40         3003       5003.0
9   70012.0     250.45         3002       5002.0
10  70012.0      75.29         3001       5003.0
11  70013.0    3045.60         3001       5003.0


In [11]:
print("bfill（后向填充）:")
print(df2.bfill())

bfill（后向填充）:
     ord_no  purch_amt  customer_id  salesman_id
0   70001.0     150.50         3002       5002.0
1   70002.0     270.65         3001       5003.0
2   70002.0      65.26         3001       5001.0
4   70005.0     948.50         3002       5002.0
5   70005.0    2400.60         3001       5001.0
6   70003.0    5760.00         3001       5001.0
8   70003.0    2480.40         3003       5003.0
9   70012.0     250.45         3002       5002.0
10  70013.0      75.29         3001       5003.0
11  70013.0    3045.60         3001          NaN


In [12]:
print("固定值 0 填充:")
print(df2.fillna(0))

固定值 0 填充:
     ord_no  purch_amt  customer_id  salesman_id
0   70001.0     150.50         3002       5002.0
1       0.0     270.65         3001       5003.0
2   70002.0      65.26         3001       5001.0
4       0.0     948.50         3002       5002.0
5   70005.0    2400.60         3001       5001.0
6       0.0    5760.00         3001       5001.0
8   70003.0    2480.40         3003       5003.0
9   70012.0     250.45         3002       5002.0
10      0.0      75.29         3001       5003.0
11  70013.0    3045.60         3001          0.0


In [13]:
# 按列均值填充：先算每列均值，再填
col_means = df2.mean(numeric_only=True).round(1)
print("每列均值（保留 1 位）:")
print(col_means)

print("\n用均值填充:")
print(df2.fillna(col_means))

每列均值（保留 1 位）:
ord_no         70006.0
purch_amt       1544.7
customer_id     3001.5
salesman_id     5002.0
dtype: float64

用均值填充:
     ord_no  purch_amt  customer_id  salesman_id
0   70001.0     150.50         3002       5002.0
1   70006.0     270.65         3001       5003.0
2   70002.0      65.26         3001       5001.0
4   70006.0     948.50         3002       5002.0
5   70005.0    2400.60         3001       5001.0
6   70006.0    5760.00         3001       5001.0
8   70003.0    2480.40         3003       5003.0
9   70012.0     250.45         3002       5002.0
10  70006.0      75.29         3001       5003.0
11  70013.0    3045.60         3001       5002.0


**注意**：`ffill` 只能“拿上面的值补下面”。如果某列的开头就是缺失，`ffill` 仍然会留下 `NaN`；这时可以配合 `bfill`，或者直接 `fillna`。

In [14]:
# 小例子：第一行是缺失，ffill 补不到
demo = pd.DataFrame({"x": [np.nan, 1, np.nan, 3]})
print("原始:")
print(demo)
print("\nffill 后:")
print(demo.ffill())
print("\n剩余缺失个数:", demo.ffill().isna().sum().to_dict())

原始:
     x
0  NaN
1  1.0
2  NaN
3  3.0

ffill 后:
     x
0  NaN
1  1.0
2  1.0
3  3.0

剩余缺失个数: {'x': 1}


In [15]:
# inplace=True 会返回 None，不要把返回值赋给变量
tmp = df2.copy()
result = tmp.fillna(0, inplace=True)
print("fillna(..., inplace=True) 的返回值:", result)
print(tmp.head(3))

fillna(..., inplace=True) 的返回值:      ord_no  purch_amt  customer_id  salesman_id
0   70001.0     150.50         3002       5002.0
1       0.0     270.65         3001       5003.0
2   70002.0      65.26         3001       5001.0
4       0.0     948.50         3002       5002.0
5   70005.0    2400.60         3001       5001.0
6       0.0    5760.00         3001       5001.0
8   70003.0    2480.40         3003       5003.0
9   70012.0     250.45         3002       5002.0
10      0.0      75.29         3001       5003.0
11  70013.0    3045.60         3001          0.0
    ord_no  purch_amt  customer_id  salesman_id
0  70001.0     150.50         3002       5002.0
1      0.0     270.65         3001       5003.0
2  70002.0      65.26         3001       5001.0


> **建议**：少用 `inplace=True`。写成 `df2 = df2.fillna(0)` 更清楚，也避免“为什么我的变量变成 None”这类错误。

## 4. 完整案例：订单数据五步处理

对应课件开头的 Exercise：

1. 统计每列缺失个数；
2. 删除 `ord_date`，保存为 `df1`；
3. 删除缺失值超过 1 个的行，保存为 `df2`；
4. 基于 `df2` 做前向填充，保存为 `df3`，不修改 `df2`；
5. 基于 `df2` 用列均值（保留 1 位小数）填充，保存为 `df4`。

In [16]:
df1 = df.drop(columns='ord_date')
df2 = df1.dropna(thresh=3)
df3 = df2.ffill()
df4 = df2.fillna(df2.mean(numeric_only=True).round(1))

print("df1 shape:", df1.shape, "| df2 shape:", df2.shape)
print("df2 缺失:\n", df2.isna().sum())
print("\ndf3 缺失:\n", df3.isna().sum())
print("\ndf4 缺失:\n", df4.isna().sum())

df1 shape: (12, 4) | df2 shape: (10, 4)
df2 缺失:
 ord_no         4
purch_amt      0
customer_id    0
salesman_id    1
dtype: int64

df3 缺失:
 ord_no         0
purch_amt      0
customer_id    0
salesman_id    0
dtype: int64

df4 缺失:
 ord_no         0
purch_amt      0
customer_id    0
salesman_id    0
dtype: int64


In [17]:
print("df3（ffill 后）:")
print(df3)

df3（ffill 后）:
     ord_no  purch_amt  customer_id  salesman_id
0   70001.0     150.50         3002       5002.0
1   70001.0     270.65         3001       5003.0
2   70002.0      65.26         3001       5001.0
4   70002.0     948.50         3002       5002.0
5   70005.0    2400.60         3001       5001.0
6   70005.0    5760.00         3001       5001.0
8   70003.0    2480.40         3003       5003.0
9   70012.0     250.45         3002       5002.0
10  70012.0      75.29         3001       5003.0
11  70013.0    3045.60         3001       5003.0


In [18]:
print("df4（均值填充后）:")
print(df4)

df4（均值填充后）:
     ord_no  purch_amt  customer_id  salesman_id
0   70001.0     150.50         3002       5002.0
1   70006.0     270.65         3001       5003.0
2   70002.0      65.26         3001       5001.0
4   70006.0     948.50         3002       5002.0
5   70005.0    2400.60         3001       5001.0
6   70006.0    5760.00         3001       5001.0
8   70003.0    2480.40         3003       5003.0
9   70012.0     250.45         3002       5002.0
10  70006.0      75.29         3001       5003.0
11  70013.0    3045.60         3001       5002.0


**注意**：对 `ord_no`、`customer_id`、`salesman_id` 这种“编号”做均值填充在业务上没有意义。均值填充只适合连续数值变量；编号更适合删除、前向填充，或保留缺失。

## 5. 为什么需要 MultiIndex

普通 DataFrame 是二维的：行索引 + 列。现实数据经常有更多层级，例如“州 + 年份”“学生 + 科目”。MultiIndex 把多个层级放进索引里，仍然保持二维表结构。

先看“用 tuple 当普通索引”的做法。

In [19]:
index_tuples = [('California', '2000'), ('California', '2010'),
                ('New York', '2000'), ('New York', '2010'),
                ('Texas', '2000'), ('Texas', '2010')]

data = {'populations': [33871648, 37253956,
                        18976457, 19378102,
                        20851820, 25145561],
        'income': [52396, 69562,
                   61297, 70312,
                   49995, 55340]}

pop = pd.DataFrame(data, index=index_tuples)
print(pop)
print("index 类型:", type(pop.index))

                    populations  income
(California, 2000)     33871648   52396
(California, 2010)     37253956   69562
(New York, 2000)       18976457   61297
(New York, 2010)       19378102   70312
(Texas, 2000)          20851820   49995
(Texas, 2010)          25145561   55340
index 类型: <class 'pandas.Index'>


In [20]:
try:
    pop.loc['California']
except KeyError as e:
    print("pop.loc['California'] -> KeyError:", e)

pop.loc['California'] -> KeyError: 'California'


普通 `Index` 里的 tuple 只是“一个整体”，pandas 不会自动按第一层筛选。可以手动写列表推导式：

In [21]:
print(pop.loc[[('California', '2010')]])
print()
print(pop.loc[[i for i in pop.index if i[0] == 'California']])

                    populations  income
(California, 2010)     37253956   69562

                    populations  income
(California, 2000)     33871648   52396
(California, 2010)     37253956   69562


转成真正的 `MultiIndex` 后，就可以按层级选择。

In [22]:
multi_index = pd.MultiIndex.from_tuples(index_tuples, names=['State', 'Year'])
pop = pop.reindex(multi_index)

print(pop)
print("index 类型:", type(pop.index))
print("index names:", pop.index.names)

                 populations  income
State      Year                     
California 2000     33871648   52396
           2010     37253956   69562
New York   2000     18976457   61297
           2010     19378102   70312
Texas      2000     20851820   49995
           2010     25145561   55340
index 类型: <class 'pandas.MultiIndex'>
index names: ['State', 'Year']


## 6. 创建 MultiIndex

三种最常用的构造方式：

| 方法 | 输入 |
|---|---|
| `from_arrays` | 每个层级的取值列表 |
| `from_tuples` | 每个完整索引 tuple |
| `from_product` | 各层取值的笛卡尔积 |

In [23]:
print(pd.MultiIndex.from_arrays(
    [['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
    names=['letter', 'number']))

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           names=['letter', 'number'])


In [24]:
print(pd.MultiIndex.from_tuples(
    [('a', 1), ('a', 2), ('b', 1), ('b', 2)],
    names=['letter', 'number']))

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           names=['letter', 'number'])


In [25]:
print(pd.MultiIndex.from_product(
    [['a', 'b'], [1, 2]],
    names=['letter', 'number']))

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           names=['letter', 'number'])


`from_product` 很适合“所有组合”的场景，例如 2013/2014 两年 × 两次 visit。

In [26]:
# Series：用 tuple 作为字典的 key
data = {('California', 2000): 33871648,
        ('California', 2010): 37253956,
        ('Texas', 2000): 20851820,
        ('Texas', 2010): 25145561,
        ('New York', 2000): 18976457,
        ('New York', 2010): 19378102}

pop_series = pd.Series(data)
print(pop_series)
print("index names:", pop_series.index.names)

California  2000    33871648
            2010    37253956
Texas       2000    20851820
            2010    25145561
New York    2000    18976457
            2010    19378102
dtype: int64
index names: [None, None]


In [27]:
# DataFrame：index 传“每个层级的列表”
df_mi = pd.DataFrame(
    np.arange(8).reshape(4, 2),
    index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
    columns=['data1', 'data2']
)
print(df_mi)

     data1  data2
a 1      0      1
  2      2      3
b 1      4      5
  2      6      7


给索引层级命名，能让结果表头和 `loc` 更可读。

In [28]:
pop_series.index.names = ['state', 'year']
print(pop_series)

state       year
California  2000    33871648
            2010    37253956
Texas       2000    20851820
            2010    25145561
New York    2000    18976457
            2010    19378102
dtype: int64


## 7. MultiIndex 的选择与切片

`loc` 的规则：

- 只给第一层：`pop.loc['California']`，返回该层下的所有行；
- 给完整 tuple：`pop.loc[('California', '2010')]`，返回一行；
- 给 tuple 的列表：`pop.loc[[('California', '2010')]]`，保留 DataFrame；
- 按第二层选：不能直接 `pop.loc['2010']`，要用 `xs(..., level=...)`。

In [29]:
print(pop.loc['California'])

      populations  income
Year                     
2000     33871648   52396
2010     37253956   69562


In [30]:
print(pop.loc[('California', '2010')])

populations    37253956
income            69562
Name: (California, 2010), dtype: int64


In [31]:
print(pop.loc[[('California', '2010')]])

                 populations  income
State      Year                     
California 2010     37253956   69562


In [32]:
try:
    pop.loc['2010']
except KeyError as e:
    print("pop.loc['2010'] -> KeyError:", e)

pop.loc['2010'] -> KeyError: '2010'


In [33]:
# xs = cross-section：按指定层级取值
print(pop.xs('2010', level='Year'))

            populations  income
State                          
California     37253956   69562
New York       19378102   70312
Texas          25145561   55340


In [34]:
# 布尔筛选同样可用
mask = pop['income'] > 60000
print(mask)
print(pop.loc[mask])

State       Year
California  2000    False
            2010     True
New York    2000     True
            2010     True
Texas       2000    False
            2010    False
Name: income, dtype: bool
                 populations  income
State      Year                     
California 2010     37253956   69562
New York   2000     18976457   61297
           2010     19378102   70312


**注意**：`loc['California']` 返回的是“去掉第一层索引后的子表”；`loc[('California','2010')]` 返回的是 Series（一行）。想看形状就用 `.shape`。

## 8. 排序、`IndexSlice` 与复杂选择

MultiIndex 的切片要求索引是**有序的**。未排序时直接切片会报 `UnsortedIndexError`。

In [35]:
data_unsorted = pd.Series(
    np.random.default_rng(0).random(6),
    index=pd.MultiIndex.from_product([['a', 'c', 'b'], [1, 2]],
                                     names=['char', 'int'])
)
print(data_unsorted)

char  int
a     1      0.636962
      2      0.269787
c     1      0.040974
      2      0.016528
b     1      0.813270
      2      0.912756
dtype: float64


In [36]:
try:
    data_unsorted['a':'b']
except Exception as e:
    print(type(e).__name__, ":", e)

UnsortedIndexError : 'Key length (1) was greater than MultiIndex lexsort depth (0)'


In [37]:
data_sorted = data_unsorted.sort_index()
print(data_sorted)
print()
print(data_sorted['a':'b'])

char  int
a     1      0.636962
      2      0.269787
b     1      0.813270
      2      0.912756
c     1      0.040974
      2      0.016528
dtype: float64

char  int
a     1      0.636962
      2      0.269787
b     1      0.813270
      2      0.912756
dtype: float64


`pd.IndexSlice` 是 `.loc` 的好帮手，尤其当行和列都需要“第一层全部、第二层指定值”时。

In [38]:
idx = pd.IndexSlice

print(pop.sort_index().loc[idx['California':'New York', '2010'], :])

                 populations  income
State      Year                     
California 2010     37253956   69562
New York   2010     19378102   70312


In [39]:
print(pop.sort_index().loc[idx[:, '2000'], :])

                 populations  income
State      Year                     
California 2000     33871648   52396
New York   2000     18976457   61297
Texas      2000     20851820   49995

等价写法是 `slice(None)`，但 `IndexSlice` 更易读：

```python
pop.loc[(slice(None), '2000'), :]
```

> **pandas 3 小提示**：行索引是 tuple 时，`.loc[idx[:, '2000']]` 可能把第二个元素误当成列名。把列位置写清楚（末尾加 `, :`）最稳。

## 9. `stack` / `unstack`：宽表与长表

- `stack()`：把**列**“压”到行索引里 → 变长（wide → long），结果通常是 MultiIndex Series。
- `unstack()`：把**行索引的某一层**“抬”到列上 → 变宽（long → wide），结果通常是 DataFrame。

In [40]:
df_wide = pd.DataFrame({
    'A': [1, 2, 3],
    'B': [4, 5, 6],
    'C': [7, 8, 9]
}, index=['X', 'Y', 'Z'])

print(df_wide)
print("\nstack():")
print(df_wide.stack())

   A  B  C
X  1  4  7
Y  2  5  8
Z  3  6  9

stack():


X  A    1
   B    4
   C    7
Y  A    2
   B    5
   C    8
Z  A    3
   B    6
   C    9
dtype: int64


In [41]:
print("stack().unstack():")
print(df_wide.stack().unstack())

stack().unstack():


   A  B  C
X  1  4  7
Y  2  5  8
Z  3  6  9


In [42]:
print("df.unstack():")
print(df_wide.unstack())

df.unstack():
A  X    1
   Y    2
   Z    3
B  X    4
   Y    5
   Z    6
C  X    7
   Y    8
   Z    9
dtype: int64


对 `pop` 这种已经有 MultiIndex 的表，`unstack(level=...)` 决定把哪一层行索引变成列。

In [43]:
print("pop:")
print(pop)
print("\nunstack(level=0):")
print(pop.unstack(level=0))

pop:
                 populations  income
State      Year                     
California 2000     33871648   52396
           2010     37253956   69562
New York   2000     18976457   61297
           2010     19378102   70312
Texas      2000     20851820   49995
           2010     25145561   55340

unstack(level=0):
      populations                         income                
State  California  New York     Texas California New York  Texas
Year                                                            
2000     33871648  18976457  20851820      52396    61297  49995
2010     37253956  19378102  25145561      69562    70312  55340


In [44]:
print("unstack(level=1):")
print(pop.unstack(level=1))

unstack(level=1):
           populations           income       
Year              2000      2010   2000   2010
State                                         
California    33871648  37253956  52396  69562
New York      18976457  19378102  61297  70312
Texas         20851820  25145561  49995  55340


In [45]:
print("unstack() 默认 level=1:")
print(pop.unstack())
print("\nunstack(level=1).stack() 回到原样:")
print(pop.unstack(level=1).stack())

unstack() 默认 level=1:
           populations           income       
Year              2000      2010   2000   2010
State                                         
California    33871648  37253956  52396  69562
New York      18976457  19378102  61297  70312
Texas         20851820  25145561  49995  55340

unstack(level=1).stack() 回到原样:
                 populations  income
State      Year                     
California 2000     33871648   52396
           2010     37253956   69562
New York   2000     18976457   61297
           2010     19378102   70312
Texas      2000     20851820   49995
           2010     25145561   55340


In [46]:
print("unstack(level=0).stack() 交换了两层索引:")
print(pop.unstack(level=0).stack())

unstack(level=0).stack() 交换了两层索引:
                 populations  income
Year State                          
2000 California     33871648   52396
     New York       18976457   61297
     Texas          20851820   49995
2010 California     37253956   69562
     New York       19378102   70312
     Texas          25145561   55340


**注意**：pandas 3 的新版 `stack()` 会保留缺失值，而且不再接受 `dropna` 参数（旧版本行为不同）。如果看到“某些组合消失了”，先检查 pandas 版本和 `stack` 的实现。

## 10. 列的 MultiIndex

MultiIndex 不只可以用于行，也可以用于列。这样一张表就能同时表达“科目 × 指标”。

In [47]:
index = pd.MultiIndex.from_product(
    [[2013, 2014], [1, 2]],
    names=['year', 'visit']
)
columns = pd.MultiIndex.from_product(
    [['Bob', 'Guido', 'Sue'], ['HR', 'Temp']],
    names=['subject', 'type']
)

print(index)
print(columns)

MultiIndex([(2013, 1),
            (2013, 2),
            (2014, 1),
            (2014, 2)],
           names=['year', 'visit'])
MultiIndex([(  'Bob',   'HR'),
            (  'Bob', 'Temp'),
            ('Guido',   'HR'),
            ('Guido', 'Temp'),
            (  'Sue',   'HR'),
            (  'Sue', 'Temp')],
           names=['subject', 'type'])


In [48]:
rng = np.random.default_rng(0)
data = np.round(rng.normal(size=(4, 6)), 1)
data[:, ::2] *= 10          # 让 HR 列和 Temp 列量级不同
data += 37

health_data = pd.DataFrame(data, index=index, columns=columns)
print(health_data)

subject      Bob       Guido         Sue      
type          HR  Temp    HR  Temp    HR  Temp
year visit                                    
2013 1      38.0  36.9  43.0  37.1  32.0  37.4
     2      50.0  37.9  30.0  35.7  31.0  37.0
2014 1      14.0  36.8  25.0  36.3  32.0  36.7
     2      41.0  38.0  36.0  38.4  30.0  37.4


选择列时也要按层级给出 tuple：

In [49]:
print("health_data['Guido', 'HR']:")
print(health_data['Guido', 'HR'])

health_data['Guido', 'HR']:
year  visit
2013  1        43.0
      2        30.0
2014  1        25.0
      2        36.0
Name: (Guido, HR), dtype: float64


In [50]:
print("iloc[:2, 2:4]:")
print(health_data.iloc[:2, 2:4])

iloc[:2, 2:4]:
subject    Guido      
type          HR  Temp
year visit            
2013 1      43.0  37.1
     2      30.0  35.7


In [51]:
print("loc[:, ('Bob', 'Temp')]:")
print(health_data.loc[:, ('Bob', 'Temp')])

loc[:, ('Bob', 'Temp')]:
year  visit
2013  1        36.9
      2        37.9
2014  1        36.8
      2        38.0
Name: (Bob, Temp), dtype: float64


**注意**：`.loc` 的行、列索引如果都是 tuple，不能直接写 `(:, 1)` 这种 Python 语法；要么用 `slice(None)`，要么用 `pd.IndexSlice`。

In [52]:
try:
    eval("health_data.loc[(:, 1), (:, 'HR')]")
except SyntaxError as e:
    print("直接写 (:, 1) -> SyntaxError:", e)

直接写 (:, 1) -> SyntaxError: invalid syntax (<string>, line 1)


In [53]:
idx = pd.IndexSlice

print("所有 visit=1 的 Bob/Guido/Sue 的 HR:")
print(health_data.loc[idx[:, 1], idx[:, 'HR']])

所有 visit=1 的 Bob/Guido/Sue 的 HR:
subject      Bob Guido   Sue
type          HR    HR    HR
year visit                  
2013 1      38.0  43.0  32.0
2014 1      14.0  25.0  32.0


## 11. `reset_index` / `set_index`

- `reset_index()`：把索引层级变成普通列；
- `set_index(['State', 'Year'])`：把普通列变成 MultiIndex。

In [54]:
print(pop.reset_index(level=1))

            Year  populations  income
State                                
California  2000     33871648   52396
California  2010     37253956   69562
New York    2000     18976457   61297
New York    2010     19378102   70312
Texas       2000     20851820   49995
Texas       2010     25145561   55340


In [55]:
pop_flat = pop.reset_index()
print(pop_flat)
print("columns:", list(pop_flat.columns))

        State  Year  populations  income
0  California  2000     33871648   52396
1  California  2010     37253956   69562
2    New York  2000     18976457   61297
3    New York  2010     19378102   70312
4       Texas  2000     20851820   49995
5       Texas  2010     25145561   55340
columns: ['State', 'Year', 'populations', 'income']


In [56]:
print(pop_flat.set_index(['State', 'Year']))

                 populations  income
State      Year                     
California 2000     33871648   52396
           2010     37253956   69562
New York   2000     18976457   61297
           2010     19378102   70312
Texas      2000     20851820   49995
           2010     25145561   55340


**注意**：`reset_index()` 默认把所有索引层都变成列；只想放回一层就写 `level=...`。`set_index` 默认会删除原来的列（`drop=True`），想保留就写 `drop=False`。

## 12. `groupby` 基础

`groupby` 是“split → apply → combine”：

1. 按 key 把行拆成若干组；
2. 对每组做聚合 / 变换 / 过滤；
3. 把结果拼回一张表。

In [57]:
sales = pd.DataFrame({
    'Region': ['North', 'North', 'South', 'South', 'North', 'South'],
    'Category': ['A', 'B', 'A', 'B', 'A', 'B'],
    'Sales': [100, 150, 200, 250, 300, 350],
    'Profit': [20, 30, 40, 50, 60, 70]
})
print(sales)

  Region Category  Sales  Profit
0  North        A    100      20
1  North        B    150      30
2  South        A    200      40
3  South        B    250      50
4  North        A    300      60
5  South        B    350      70


In [58]:
print(sales.groupby('Category')['Sales'].sum())
print()
print(sales.groupby('Category').sum(numeric_only=True))

Category
A    600
B    750
Name: Sales, dtype: int64

          Sales  Profit
Category               
A           600     120
B           750     150


In [59]:
print("每组的行数:")
print(sales.groupby('Category').size())
print("\nProfit 的非缺失个数:")
print(sales.groupby('Category')['Profit'].count())

每组的行数:
Category
A    3
B    3
dtype: int64

Profit 的非缺失个数:
Category
A    3
B    3
Name: Profit, dtype: int64


**注意**：对包含字符串列的 DataFrame 直接 `.sum()`，pandas 会把字符串拼起来。做数值聚合时加 `numeric_only=True`，或先选数值列。

## 13. 多列分组与 `as_index`

按多列分组时，默认这些 key 会变成结果的 MultiIndex。加 `as_index=False` 可以把它们变回普通列。

In [60]:
multi_group = sales.groupby(['Region', 'Category']).sum(numeric_only=True)
print(multi_group)
print("index:", multi_group.index.names)

                 Sales  Profit
Region Category               
North  A           400      80
       B           150      30
South  A           200      40
       B           600     120
index: ['Region', 'Category']


In [61]:
print(sales.groupby(['Region', 'Category'], as_index=False).sum(numeric_only=True))

  Region Category  Sales  Profit
0  North        A    400      80
1  North        B    150      30
2  South        A    200      40
3  South        B    600     120


**选择建议**：
- 后面还要用 `loc` 按 key 选 → 保留索引（默认）；
- 要直接导出成普通表 / 继续 `merge` → 用 `as_index=False` 或 `reset_index()`。

## 14. `agg`：一次做多种聚合

`agg` 可以给不同列指定不同函数，也可以给同一列指定多个函数。

In [62]:
result = sales.groupby(['Region', 'Category']).agg({
    'Sales': ['mean', 'sum', 'std'],
    'Profit': ['min', 'max', 'count']
})
print(result)
print("\ncolumns 是 MultiIndex:")
print(result.columns)

                 Sales                  Profit          
                  mean  sum         std    min max count
Region Category                                         
North  A         200.0  400  141.421356     20  60     2
       B         150.0  150         NaN     30  30     1
South  A         200.0  200         NaN     40  40     1
       B         300.0  600   70.710678     50  70     2

columns 是 MultiIndex:
MultiIndex([( 'Sales',  'mean'),
            ( 'Sales',   'sum'),
            ( 'Sales',   'std'),
            ('Profit',   'min'),
            ('Profit',   'max'),
            ('Profit', 'count')],
           )


In [63]:
print("取 Sales 的 mean 列:")
print(result[('Sales', 'mean')])

取 Sales 的 mean 列:
Region  Category
North   A           200.0
        B           150.0
South   A           200.0
        B           300.0
Name: (Sales, mean), dtype: float64


In [64]:
# 命名聚合：结果列名更清晰
print(sales.groupby('Category').agg(
    total_sales=('Sales', 'sum'),
    avg_profit=('Profit', 'mean'),
    max_profit=('Profit', 'max')
))

          total_sales  avg_profit  max_profit
Category                                     
A                 600        40.0          60
B                 750        50.0          70


In [65]:
print(sales.groupby('Category').describe())

         Sales                                                  Profit                                          
         count   mean    std    min    25%    50%    75%    max  count  mean   std   min   25%   50%   75%   max
Category                                                                                                        
A          3.0  200.0  100.0  100.0  150.0  200.0  250.0  300.0    3.0  40.0  20.0  20.0  30.0  40.0  50.0  60.0
B          3.0  250.0  100.0  150.0  200.0  250.0  300.0  350.0    3.0  50.0  20.0  30.0  40.0  50.0  60.0  70.0


## 15. 遍历 groupby 对象

`groupby` 返回的是 `DataFrameGroupBy` 对象（可迭代），每次迭代得到 `(组名, 子表)`。

In [66]:
gp = sales.groupby('Category')
print(gp)
print()

for name, group in gp:
    print("组名:", name)
    print(group)
    print()


组名: A
  Region Category  Sales  Profit
0  North        A    100      20
2  South        A    200      40
4  North        A    300      60

组名: B
  Region Category  Sales  Profit
1  North        B    150      30
3  South        B    250      50
5  South        B    350      70



**用途**：需要对每组画图、分别建模、或写自定义逻辑时，遍历很直观；但如果只是求和/均值，优先用 `agg`，更快也更简洁。

## 16. 自定义分组键：mapping / function / MultiIndex

`groupby` 的 key 不一定非得是列名，也可以是：

- 一个把索引值映射到组名的 dict / Series；
- 一个函数（作用于索引）；
- 多个 key 组成的列表。

In [67]:
key_df = pd.DataFrame({
    'key': ['A', 'B', 'C', 'A', 'B', 'C'],
    'data': range(6),
    'random': np.random.default_rng(1).random(6)
}).set_index('key')

print(key_df)
print()
print(key_df.groupby('key').sum())

     data    random
key                
A       0  0.511822
B       1  0.950464
C       2  0.144160
A       3  0.948649
B       4  0.311831
C       5  0.423326

     data    random
key                
A       3  1.460471
B       5  1.262295
C       7  0.567486


In [68]:
mapping = {'A': 'vowel', 'B': 'consonant', 'C': 'consonant'}
print(key_df.groupby(mapping).sum())

           data    random
key                      
consonant    12  1.829781
vowel         3  1.460471


In [69]:
print(key_df.groupby(str.lower).mean())

     data    random
key                
a     1.5  0.730236
b     2.5  0.631148
c     3.5  0.283743


In [70]:
print(key_df.groupby([str.lower, mapping]).mean())

               data    random
key key                      
a   vowel       1.5  0.730236
b   consonant   2.5  0.631148
c   consonant   3.5  0.283743


**注意**：用函数分组时，函数作用在**索引**上；如果 key 还是普通列，要先 `set_index`，或者用 `groupby(df['col'].str.lower())`。

## 17. 练习：health_data 的层级交换与选择

(1) 交换 `visit` 和 `year`，保存为 `health_data2`。

(2) 取出 Bob 和 Sue 的 HR 数据，只要每年的第一次 visit（visit=1）。

<details>
<summary>点击查看答案</summary>

```python
health_data2 = health_data.unstack(level=0).stack()
idx = pd.IndexSlice
health_data.loc[idx[:, 1], idx[['Bob', 'Sue'], 'HR']]
```

</details>

In [71]:
health_data2 = health_data.unstack(level=0).stack()
print(health_data2)
print("index names:", health_data2.index.names)

subject      Bob       Guido         Sue      
type          HR  Temp    HR  Temp    HR  Temp
visit year                                    
1     2013  38.0  36.9  43.0  37.1  32.0  37.4
      2014  14.0  36.8  25.0  36.3  32.0  36.7
2     2013  50.0  37.9  30.0  35.7  31.0  37.0
      2014  41.0  38.0  36.0  38.4  30.0  37.4
index names: ['visit', 'year']


In [72]:
idx = pd.IndexSlice
print(health_data.loc[idx[:, 1], idx[['Bob', 'Sue'], 'HR']])

subject      Bob   Sue
type          HR    HR
year visit            
2013 1      38.0  32.0
2014 1      14.0  32.0


## 18. 练习：sales 的 MultiIndex 构造

(1) 用 `sales_arrays` 创建名为 `sales` / `city` 的 MultiIndex。

(2) 用这个 MultiIndex 作为行索引、`column_names` 作为列，构造 DataFrame `df`。

(3) 用 `observations` 的第一列构造 Series `df2`。

(4) 把 `df2` 的索引信息变成列，保存为 `df3`，数值列命名为 `obs`。

<details>
<summary>点击查看答案</summary>

```python
sales_tuples = list(zip(sales_arrays[0], sales_arrays[1]))
sales_index = pd.MultiIndex.from_tuples(sales_tuples, names=['sales', 'city'])

df  = pd.DataFrame(observations, index=sales_index, columns=column_names)
df2 = pd.Series(observations[:, 0], index=sales_index)
df3 = df2.reset_index(name='obs')
```

</details>

In [73]:
sales_arrays = [['sale1', 'sale1', 'sale2', 'sale2', 'sale3', 'sale3', 'sale4', 'sale4'],
                ['city1', 'city2', 'city1', 'city2', 'city1', 'city2', 'city1', 'city2']]
observations = np.random.default_rng(7).normal(size=(8, 5))
column_names = ['obs1', 'obs2', 'obs3', 'obs4', 'obs5']

sales_tuples = list(zip(sales_arrays[0], sales_arrays[1]))
sales_index = pd.MultiIndex.from_tuples(sales_tuples, names=['sales', 'city'])
print(sales_index)

MultiIndex([('sale1', 'city1'),
            ('sale1', 'city2'),
            ('sale2', 'city1'),
            ('sale2', 'city2'),
            ('sale3', 'city1'),
            ('sale3', 'city2'),
            ('sale4', 'city1'),
            ('sale4', 'city2')],
           names=['sales', 'city'])


In [74]:
df_sales = pd.DataFrame(observations, index=sales_index, columns=column_names)
print(df_sales)

                 obs1      obs2      obs3      obs4      obs5
sales city                                                   
sale1 city1  0.001230  0.298746 -0.274138 -0.890592 -0.454671
      city2 -0.991647  0.060144  1.340215 -0.492207 -0.620475
sale2 city1  0.489842  0.356887  0.105414 -0.930468 -0.029252
      city2  0.695303 -1.344215 -0.457616 -1.901223 -1.289538
sale3 city1 -1.841735 -0.235091 -1.267446  0.271264  0.156751
      city2 -0.186931 -2.516760 -0.538693 -0.048501  0.113309
sale4 city1 -1.530136 -0.477753 -0.978519 -0.808837  1.060899
      city2 -0.807535 -0.032522  0.884390 -0.583600 -0.111702


In [75]:
df2 = pd.Series(observations[:, 0], index=sales_index)
print(df2)

sales  city 
sale1  city1    0.001230
       city2   -0.991647
sale2  city1    0.489842
       city2    0.695303
sale3  city1   -1.841735
       city2   -0.186931
sale4  city1   -1.530136
       city2   -0.807535
dtype: float64


In [76]:
df3 = df2.reset_index(name='obs')
print(df3)

   sales   city       obs
0  sale1  city1  0.001230
1  sale1  city2 -0.991647
2  sale2  city1  0.489842
3  sale2  city2  0.695303
4  sale3  city1 -1.841735
5  sale3  city2 -0.186931
6  sale4  city1 -1.530136
7  sale4  city2 -0.807535


## 19. 练习：学生数据分组与 BMI

(1) 按 `school_code` 和 `class` 分组，求 `age` 的 mean / min / max。

(2) 只考虑 age ≤ 13 的学生，按 `school_code` 分组，求 BMI 的平均值。BMI = weight / (height/100)^2。

<details>
<summary>点击查看答案</summary>

```python
student_data.groupby(['school_code', 'class'])['age'].agg(['mean', 'min', 'max'])

student_data['BMI'] = student_data['weight'] / (student_data['height'] / 100) ** 2
student_data[student_data['age'] <= 13].groupby('school_code')['BMI'].mean()
```

</details>

In [77]:
student_data = pd.DataFrame({
    'school_code': ['s001', 's002', 's003', 's001', 's002', 's004'],
    'class': ['V', 'V', 'VI', 'VI', 'V', 'VI'],
    'name': ['Alberto Franco', 'Gino Mcneill', 'Ryan Parkes',
             'Eesha Hinton', 'Gino Mcneill', 'David Parkes'],
    'date_Of_Birth ': ['15/05/2002', '17/05/2002', '16/02/1999',
                       '25/09/1998', '11/05/2002', '15/09/1997'],
    'age': [12, 12, 13, 13, 14, 12],
    'height': [173, 192, 186, 167, 151, 159],
    'weight': [70, 85, 93, 65, 50, 60],
    'address': ['street1', 'street2', 'street3', 'street1', 'street2', 'street4']
}, index=['S1', 'S2', 'S3', 'S4', 'S5', 'S6'])

print(student_data)

   school_code class            name date_Of_Birth   age  height  weight  address
S1        s001     V  Alberto Franco     15/05/2002   12     173      70  street1
S2        s002     V    Gino Mcneill     17/05/2002   12     192      85  street2
S3        s003    VI     Ryan Parkes     16/02/1999   13     186      93  street3
S4        s001    VI    Eesha Hinton     25/09/1998   13     167      65  street1
S5        s002     V    Gino Mcneill     11/05/2002   14     151      50  street2
S6        s004    VI    David Parkes     15/09/1997   12     159      60  street4


In [78]:
print(student_data.groupby(['school_code', 'class'])['age'].agg(['mean', 'min', 'max']))

                   mean  min  max
school_code class                
s001        V      12.0   12   12
            VI     13.0   13   13
s002        V      13.0   12   14
s003        VI     13.0   13   13
s004        VI     12.0   12   12


In [79]:
student_data['BMI'] = student_data['weight'] / (student_data['height'] / 100) ** 2
print(student_data[['school_code', 'age', 'BMI']])

   school_code  age        BMI
S1        s001   12  23.388687
S2        s002   12  23.057726
S3        s003   13  26.881720
S4        s001   13  23.306680
S5        s002   14  21.928863
S6        s004   12  23.733238


In [80]:
avg_bmi = (student_data[student_data['age'] <= 13]
           .groupby('school_code')['BMI']
           .mean())
print(avg_bmi)

school_code
s001    23.347683
s002    23.057726
s003    26.881720
s004    23.733238
Name: BMI, dtype: float64


## 20. 速查表与常见坑

### 20.1 缺失值

| 想要 | 写法 |
|---|---|
| 检测缺失 | `df.isna()` / `df.isnull()` |
| 每列缺失数 | `df.isna().sum()` |
| 每行缺失数 | `df.isna().sum(axis=1)` |
| 删除有缺失的行 | `df.dropna()` |
| 删除缺失超过 n 个的行 | `df.dropna(thresh=df.shape[1] - n)` |
| 删除缺失比例过高的列 | `df.dropna(axis=1, thresh=...)` |
| 前向 / 后向填充 | `df.ffill()` / `df.bfill()` |
| 指定列填不同值 | `df.fillna({'A': 0, 'B': df['B'].mean()})` |
| 插值 | `df.interpolate()` |

### 20.2 MultiIndex

| 想要 | 写法 |
|---|---|
| 从 tuple 创建 | `pd.MultiIndex.from_tuples(...)` |
| 从各层数组创建 | `pd.MultiIndex.from_arrays(...)` |
| 笛卡尔积 | `pd.MultiIndex.from_product(...)` |
| 命名层级 | `mi.names = ['State', 'Year']` |
| 选第一层 | `df.loc['California']` |
| 选完整 tuple | `df.loc[('California', '2010')]` |
| 按第二层选 | `df.xs('2010', level='Year')` |
| 复杂切片 | `df.loc[pd.IndexSlice[:, '2010'], :]` |
| 排序 | `df.sort_index()` |
| 宽转长 | `df.stack()` |
| 长转宽 | `df.unstack(level=...)` |
| 索引变列 | `df.reset_index()` |
| 列变索引 | `df.set_index(['State', 'Year'])` |

### 20.3 GroupBy

| 想要 | 写法 |
|---|---|
| 单列分组 | `df.groupby('Category')` |
| 多列分组 | `df.groupby(['Region', 'Category'])` |
| 保留普通列 | `as_index=False` 或 `reset_index()` |
| 单列聚合 | `df.groupby('k')['v'].sum()` |
| 多函数聚合 | `.agg(['mean', 'sum'])` |
| 不同列不同函数 | `.agg({'Sales': 'sum', 'Profit': 'mean'})` |
| 命名聚合 | `.agg(total=('Sales', 'sum'))` |
| 组大小 | `.size()` |
| 非缺失计数 | `.count()` |
| 遍历 | `for name, group in df.groupby('k'):` |

### 20.4 常见坑

1. **`thresh` 数的是非缺失值**，不是缺失值。
2. **`ffill` 补不了开头的缺失**；必要时配合 `bfill` 或固定值。
3. **`inplace=True` 返回 `None`**，不要写 `df = df.fillna(0, inplace=True)`。
4. **对字符串列直接 `sum()` 会拼接字符串**；数值聚合加 `numeric_only=True` 或先选列。
5. **MultiIndex 切片前要排序**，否则可能 `UnsortedIndexError`。
6. **`loc['2010']` 不能选第二层**；用 `xs('2010', level=...)`，或写清 `loc[idx[:, '2010'], :]`。
7. **pandas 3 的 `stack()` 会保留缺失值**，且不再接受 `dropna` 参数；旧版本行为不同。
8. **`reset_index()` 默认放回所有层**；只放一层用 `level=`。
9. **`groupby` 默认把 key 变成索引**；需要普通列就 `as_index=False`。
10. **`mean()` 默认跳过 `NaN`**；想严格要求非缺失用 `skipna=False`。

---

**Done.** 建议执行 `Kernel → Restart & Run All`。检查结果时先看三件事：`shape`、`index`、`dtypes`。